# Checkpoint: pipeline em uma categoria

Executa DataLoader -> extrator congelado -> decoder -> mapa de discrepancia -> escore por imagem em uma imagem defeituosa de `bottle`.

**Interpretacao:** o decoder e inicializado sem pesos treinados. Este notebook valida integracao, formatos e alinhamento imagem/mascara/mapa; o mapa nao representa desempenho de deteccao. Decisao do checkpoint: **seguir com a implementacao propria (opcao B)**; nao e necessario trocar para UniAD/anomalib nem simplificar por falha de integracao.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "dataset/train_split.csv").is_file():
    repo_root = repo_root.parent
data_root = repo_root / "dataset"
assert (data_root / "train_split.csv").is_file(), "Run the notebook from the repository."
sys.path.insert(0, str(repo_root))

import torch
from torch.utils.data import DataLoader, Subset

from src.anomaly_maps import compute_anomaly_maps, normalize_map_for_visualization
from src.anomaly_scoring import aggregate_anomaly_maps
from src.feature_extractor import WideResNetFeatureExtractor
from src.mvtec_dataset import MVTecTestDataset
from src.reconstruction_decoder import ReconstructionDecoder

category = "bottle"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Data root: {data_root}")
print(f"Device: {device}")

In [ ]:
test_dataset = MVTecTestDataset(data_root, image_size=512)
sample_index = next(
    index
    for index, (_, _, label, sample_category) in enumerate(test_dataset.samples)
    if label == 1 and sample_category == category
)
sample_loader = DataLoader(
    Subset(test_dataset, [sample_index]), batch_size=1, shuffle=False
)
images, masks, labels, categories = next(iter(sample_loader))
assert images.shape == (1, 3, 512, 512)
assert masks.shape == (1, 1, 512, 512)
print(f"Sample: {test_dataset.samples[sample_index][0]}")
print(f"Image batch: {tuple(images.shape)}; mask batch: {tuple(masks.shape)}")

In [ ]:
torch.manual_seed(42)
extractor = WideResNetFeatureExtractor(pretrained=True).to(device).eval()
decoder = ReconstructionDecoder(
    neighbor_size=7, jitter_scale=20, jitter_probability=1.0
).to(device).eval()

with torch.no_grad():
    images_device = images.to(device)
    original_features = extractor(images_device)
    reconstructed_features = decoder(original_features)
    anomaly_maps = compute_anomaly_maps(
        original_features,
        reconstructed_features,
        output_size=tuple(images.shape[-2:]),
    )
    image_scores = aggregate_anomaly_maps(
        anomaly_maps, method="topk_mean", top_fraction=0.01
    )

assert original_features.shape == reconstructed_features.shape
assert anomaly_maps.shape == masks[:, 0].shape
assert anomaly_maps.dtype == torch.float32
assert image_scores.shape == (1,) and torch.isfinite(image_scores).all()
print(f"Features: {tuple(original_features.shape)}")
print(f"Map: {tuple(anomaly_maps.shape)}, {anomaly_maps.dtype}")
print(f"Raw map range: [{anomaly_maps.min().item():.4f}, {anomaly_maps.max().item():.4f}]")
print(f"Image score (top 1% mean): {image_scores.item():.4f}")
print("Checkpoint: PASS (one sample; decoder untrained; no quality claim)")

In [ ]:
import numpy as np
from PIL import Image, ImageDraw, ImageFont

from src.mvtec_dataset import IMAGENET_MEAN, IMAGENET_STD

# Only the displayed copy is min-max normalized; anomaly_maps remains raw for metrics.
display_map = normalize_map_for_visualization(anomaly_maps)[0].cpu().numpy()
red = np.clip(1.5 - np.abs(4 * display_map - 3), 0, 1)
green = np.clip(1.5 - np.abs(4 * display_map - 2), 0, 1)
blue = np.clip(1.5 - np.abs(4 * display_map - 1), 0, 1)
heatmap = Image.fromarray(
    (np.stack((red, green, blue), axis=-1) * 255).astype(np.uint8), mode="RGB"
)

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
rgb = (images[0] * std + mean).clamp(0, 1)
rgb = Image.fromarray((rgb.permute(1, 2, 0).numpy() * 255).astype(np.uint8), mode="RGB")
mask_image = Image.fromarray((masks[0, 0].numpy() * 255).astype(np.uint8), mode="L").convert("RGB")
overlay = Image.blend(rgb, heatmap, alpha=0.45)

panel_size, margin, header = 512, 20, 86
figure = Image.new("RGB", (margin * 2 + panel_size * 3, header + panel_size + 30), "white")
draw = ImageDraw.Draw(figure)
font = ImageFont.load_default()
draw.text((margin, 12), "Bottle pipeline checkpoint - decoder untrained", fill="black", font=font)
draw.text((margin, 34), f"Top-1% image score: {image_scores.item():.4f} (smoke test only)", fill="black", font=font)
for column, (label, panel) in enumerate(
    [("Image", rgb), ("Ground-truth mask", mask_image), ("Anomaly map overlay", overlay)]
):
    left = margin + column * panel_size
    draw.text((left, 58), label, fill="black", font=font)
    figure.paste(panel, (left, header))

output_path = repo_root / "figures" / "category_pipeline_bottle.png"
output_path.parent.mkdir(parents=True, exist_ok=True)
figure.save(output_path)
print(f"Saved map print: {output_path}")
figure